# Profiling Players

In [4]:
from maverick.players import FoldBot, CallBot, AggressiveBot
from maverick import (
    PlayerLike,
    PlayerState,
    Game,
    GameEvent,
    GameEventType,
    PlayerAction,
    ActionType,
    Street
)

In [5]:
class Profiler:
    def __init__(self, game: Game = None):
        self._game = game
        self._id_to_player = {}

        self._street_action_counts = {}
        self._street_reach_counts = {}
        self._hand_counts = {}

        self.listen(game)

    def listen(self, game: Game = None) -> None:
        if game is None:
            return

        self._game = game
        self._subscribe_to_events()
        self._register_players()

    def _subscribe_to_events(self) -> None:
        for event_type in GameEventType:
            self._game.subscribe(event_type, self._handle_event)

    def _register_players(self) -> None:
        self._id_to_player = {player.id: player for player in self._game.state.players}

    def _reset_counters(self) -> None:
        players = self._game.state.players
        
        self._hand_counts = {}
        self._street_reach_counts = {}
        self._street_action_counts = {}
        
        for player in players:
            player_id = player.id
            self._hand_counts[player_id] = 0
            self._street_reach_counts[player_id] = {s: 0 for s in Street}
            self._street_action_counts[player_id] = {s: {a: 0 for a in ActionType} for s in Street}

    def _handle_event(self, event: GameEvent, game: Game) -> None:
        match event.type:
            case GameEventType.GAME_STARTED:
                self._reset_counters()

            case GameEventType.HAND_STARTED:
                for p in game.state.get_players_in_hand():
                    self._hand_counts[p.id] += 1

            case GameEventType.BETTING_ROUND_STARTED:
                for p in game.state.get_players_in_hand():
                    self._street_reach_counts[p.id][event.street] += 1

            case GameEventType.PLAYER_ACTION_TAKEN:
                action_type: ActionType = event.action.action_type
                street: Street = event.street
                for p in game.state.get_players_in_hand():
                    self._street_action_counts[p.id][street][action_type] += 1
                

In [6]:
game = Game(small_blind=10, big_blind=20, max_hands=10)

players: list[PlayerLike] = [
    CallBot(name="CallBot", state=PlayerState(stack=1000)),
    AggressiveBot(name="AggroBot", state=PlayerState(stack=1000)),
    FoldBot(name="FoldBot", state=PlayerState(stack=1000)),
]

for player in players:
    game.add_player(player)
    

profiler = Profiler(game)

game.start()

In [8]:
profiler._hand_counts

{'d56f73ada32b4559afaa9c06e7c9b458': 10,
 'f9fe592954c14ef88ba86a808168d9f6': 10,
 'cb92eff3d6d64b84886ad78c6bf8c7e9': 10}

In [9]:
profiler._street_reach_counts

{'d56f73ada32b4559afaa9c06e7c9b458': {<Street.PRE_FLOP: 0>: 10,
  <Street.FLOP: 1>: 10,
  <Street.TURN: 2>: 10,
  <Street.RIVER: 3>: 10},
 'f9fe592954c14ef88ba86a808168d9f6': {<Street.PRE_FLOP: 0>: 10,
  <Street.FLOP: 1>: 10,
  <Street.TURN: 2>: 10,
  <Street.RIVER: 3>: 10},
 'cb92eff3d6d64b84886ad78c6bf8c7e9': {<Street.PRE_FLOP: 0>: 10,
  <Street.FLOP: 1>: 0,
  <Street.TURN: 2>: 0,
  <Street.RIVER: 3>: 0}}

In [10]:
profiler._street_action_counts

{'d56f73ada32b4559afaa9c06e7c9b458': {<Street.PRE_FLOP: 0>: {<ActionType.FOLD: 1>: 10,
   <ActionType.CHECK: 2>: 0,
   <ActionType.CALL: 3>: 17,
   <ActionType.BET: 4>: 0,
   <ActionType.RAISE: 5>: 10,
   <ActionType.ALL_IN: 6>: 0},
  <Street.FLOP: 1>: {<ActionType.FOLD: 1>: 0,
   <ActionType.CHECK: 2>: 7,
   <ActionType.CALL: 3>: 10,
   <ActionType.BET: 4>: 10,
   <ActionType.RAISE: 5>: 0,
   <ActionType.ALL_IN: 6>: 0},
  <Street.TURN: 2>: {<ActionType.FOLD: 1>: 0,
   <ActionType.CHECK: 2>: 7,
   <ActionType.CALL: 3>: 10,
   <ActionType.BET: 4>: 10,
   <ActionType.RAISE: 5>: 0,
   <ActionType.ALL_IN: 6>: 0},
  <Street.RIVER: 3>: {<ActionType.FOLD: 1>: 0,
   <ActionType.CHECK: 2>: 7,
   <ActionType.CALL: 3>: 10,
   <ActionType.BET: 4>: 10,
   <ActionType.RAISE: 5>: 0,
   <ActionType.ALL_IN: 6>: 0}},
 'f9fe592954c14ef88ba86a808168d9f6': {<Street.PRE_FLOP: 0>: {<ActionType.FOLD: 1>: 10,
   <ActionType.CHECK: 2>: 0,
   <ActionType.CALL: 3>: 17,
   <ActionType.BET: 4>: 0,
   <ActionType.RA